# SE-ResNeXt-50-32x4d KL Grading

This is a single-task adaptation inspired by the Tiulpin et al. training recipe. It predicts the current five-class Kellgren-Lawrence grade only. Future progression, clinical features, and model stacking are intentionally excluded.

## Configuration Summary

| Item | Configuration |
| --- | --- |
| Task | One CNN head: KL grade 0-4 |
| Backbone | ImageNet-pretrained SE-ResNeXt-50-32x4d |
| Head | Global average pooling -> Dropout(0.50) -> Linear(2048 -> 5) |
| Input | Resize to 310, random crop to 300 during training; center-safe 300 crop for validation |
| Preprocessing | Percentile clipping 5th-99th, global [0,1] normalization, grayscale repeated to 3 channels |
| Augmentation | Gaussian noise, gamma correction, rotation +/-5 degrees, random 310 -> 300 crop |
| Loss | Five-class cross-entropy |
| Optimizer | Adam, learning rate 1e-3, weight decay 1e-4 |
| Training stages | Backbone frozen for 2 epochs, then all layers trainable for 20 epochs |
| Schedule | Learning rate reduced 10x at epoch 15 |
| Validation | Five-fold stratified subject-grouped cross-validation |
| Explainability | Post-hoc Grad-CAM can target the final convolutional block |

The original paper used longitudinal OAI/MOST radiographs and progression targets. This notebook keeps only the paper-inspired image training recipe and uses the local five-grade KL dataset.

## Run

Run every cell in a fresh Colab GPU runtime.



In [1]:
!pip -q install timm scikit-learn opencv-python-headless


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import copy
import json
import random
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score
from sklearn.model_selection import StratifiedGroupKFold

SEED = 42
DATA_ROOT = Path('/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/extracted/KneeXrayData/ClsKLData/kneeKL224')
RUN_ROOT = Path('/content/drive/MyDrive/Models/paper_se_resnext50_kl')
SOURCE_SIZE = 310
INPUT_SIZE = 300
BATCH_SIZE = 64
FROZEN_EPOCHS = 2
FULL_EPOCHS = 20
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
DROPOUT = 0.50

if not DATA_ROOT.exists():
    raise FileNotFoundError(f'KL dataset not found: {DATA_ROOT}')
RUN_ROOT.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

records = []
for path in sorted(DATA_ROOT.glob('*/*/*.png')):
    try: grade = int(path.parent.name)
    except ValueError: continue
    if grade not in range(5): continue
    # The leading filename token is used only as a grouping key. Verify it
    # against your dataset naming convention before using patient-level claims.
    subject_id = path.stem.split('_')[0]
    records.append({'image_path': str(path), 'subject_id': subject_id, 'kl_grade': grade})
frame = pd.DataFrame(records)
if frame.empty: raise RuntimeError(f'No KL PNG files found under {DATA_ROOT}')
print('Rows:', len(frame), 'subjects:', frame.subject_id.nunique(), 'device:', DEVICE)

class KLDataset(torch.utils.data.Dataset):
    def __init__(self, rows, train): self.rows=rows.reset_index(drop=True); self.train=train
    def __len__(self): return len(self.rows)
    def __getitem__(self, index):
        row=self.rows.iloc[index]
        image=cv2.imread(str(row.image_path), cv2.IMREAD_GRAYSCALE)
        if image is None: raise RuntimeError(f'Cannot decode {row.image_path}')
        image=cv2.resize(image,(SOURCE_SIZE,SOURCE_SIZE),interpolation=cv2.INTER_AREA).astype(np.float32)
        lo,hi=np.percentile(image,[5,99]); image=np.clip((image-lo)/(hi-lo+1e-8),0,1)
        if self.train:
            angle=random.uniform(-5,5); matrix=cv2.getRotationMatrix2D((SOURCE_SIZE/2,SOURCE_SIZE/2),angle,1.0)
            image=cv2.warpAffine(image,matrix,(SOURCE_SIZE,SOURCE_SIZE),borderMode=cv2.BORDER_REFLECT_101)
            image=np.power(np.clip(image,0,1),random.uniform(0.8,1.2))
            image=np.clip(image+np.random.normal(0,0.02,image.shape),0,1)
            x=random.randint(0,SOURCE_SIZE-INPUT_SIZE); y=random.randint(0,SOURCE_SIZE-INPUT_SIZE)
            image=image[y:y+INPUT_SIZE,x:x+INPUT_SIZE]
        else:
            image=image[5:305,5:305]
        tensor=torch.from_numpy(np.repeat(image[None],3,axis=0).copy()).float()
        return tensor, int(row.kl_grade)

class SEResNeXtKLCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone=timm.create_model('seresnext50_32x4d',pretrained=True,num_classes=0,global_pool='avg')
        self.classifier=nn.Sequential(nn.Dropout(DROPOUT),nn.Linear(self.backbone.num_features,5))
    def forward(self, images): return self.classifier(self.backbone(images))

def evaluate(model, loader):
    model.eval(); truth=[]; pred=[]
    with torch.no_grad():
        for images, labels in loader:
            pred.extend(model(images.to(DEVICE)).argmax(1).cpu().numpy()); truth.extend(labels.numpy())
    return {'accuracy':float(accuracy_score(truth,pred)), 'qwk':float(cohen_kappa_score(truth,pred,weights='quadratic')), 'macro_f1':float(f1_score(truth,pred,average='macro',zero_division=0))}

def train_fold(train_rows, val_rows, fold):
    model=SEResNeXtKLCNN().to(DEVICE)
    for parameter in model.backbone.parameters(): parameter.requires_grad=False
    optimizer=torch.optim.Adam(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
    train_loader=torch.utils.data.DataLoader(KLDataset(train_rows,True),batch_size=BATCH_SIZE,shuffle=True,num_workers=2,pin_memory=True)
    val_loader=torch.utils.data.DataLoader(KLDataset(val_rows,False),batch_size=BATCH_SIZE,shuffle=False,num_workers=2,pin_memory=True)
    history=[]; best=None; best_score=-float('inf')
    for epoch in range(1,FROZEN_EPOCHS+FULL_EPOCHS+1):
        if epoch==FROZEN_EPOCHS+1:
            for parameter in model.backbone.parameters(): parameter.requires_grad=True
        if epoch==15:
            for group in optimizer.param_groups: group['lr']=LEARNING_RATE*0.1
        model.train(); total=0.0; count=0
        for images, labels in train_loader:
            labels=labels.to(DEVICE); optimizer.zero_grad(set_to_none=True); loss=nn.functional.cross_entropy(model(images.to(DEVICE)),labels); loss.backward(); optimizer.step(); total+=loss.item()*len(labels); count+=len(labels)
        metrics=evaluate(model,val_loader); row={'fold':fold,'epoch':epoch,'train_loss':total/count,**metrics}; history.append(row); print(row)
        if metrics['qwk']+metrics['macro_f1'] > best_score:
            best_score=metrics['qwk']+metrics['macro_f1']; best=copy.deepcopy(model.state_dict())
    torch.save({'model_state_dict':best,'paper_inspired':'tiulpin_2019','task':'kl_grade_0_to_4','fold':fold,'config':{'batch_size':BATCH_SIZE,'frozen_epochs':FROZEN_EPOCHS,'full_epochs':FULL_EPOCHS,'learning_rate':LEARNING_RATE,'lr_drop_epoch':15,'weight_decay':WEIGHT_DECAY,'dropout':DROPOUT}}, RUN_ROOT/f'fold_{fold}_best.pth')
    return history

splitter=StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=SEED); all_history=[]
for fold,(train_idx,val_idx) in enumerate(splitter.split(frame,frame.kl_grade,groups=frame.subject_id),1):
    all_history.extend(train_fold(frame.iloc[train_idx],frame.iloc[val_idx],fold))
pd.DataFrame(all_history).to_csv(RUN_ROOT/'training_history.csv',index=False)
print('Single-head KL grading training complete:', RUN_ROOT)



Mounted at /content/drive
Rows: 9786 subjects: 9071 device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


model.safetensors: reconstructing file:   0%|          |  0.00B /  111MB            

model.safetensors: downloading bytes:           |  0.00B            

## Dataset and target

This notebook uses the local `kneeKL224` folder. Labels come from the grade directory (`0` through `4`). It does not require progression labels, clinical features, OAI/MOST follow-up data, or a second prediction head. Subject grouping uses the leading filename token as a grouping key; confirm that convention matches your files before interpreting the folds as patient-wise.

The paper's original ROI alignment and standardized 310x310 image acquisition are approximated here from the available cropped KL images by percentile normalization and resize. The training objective remains current KL grading only.



## Completion

The preceding code cell performs the five-fold single-head KL training and saves one checkpoint per fold under `RUN_ROOT`. No progression or clinical-fusion stage is included.
